In [1]:
# Add import 
import sys
import torch 
from torch import nn
from torch import optim
from prodigyopt import Prodigy # proddigy optimizer from https://github.com/konstmish/prodigy?tab=readme-ov-file

import tqdm
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")
# matplolib inline 
%matplotlib inline
import numpy as np

# allow reload of python modules
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

from dataset.RobotPathDataset.normalizer import MinMaxFeatureNormalizer

from model.models import EMA
import copy
import time

# remove all warnings
import warnings
warnings.filterwarnings("ignore")

Matplotlib - Backend: 'headless' mode detected -> use 'Agg'


In [2]:
# import gdown
# url = 'https://drive.google.com/drive/folders/16UthOas97BntH8ngVSofxgd2JrxSLDZU'
# gdown.download_folder(url)

# Hyperparams Constants

In [3]:
normalizer = MinMaxFeatureNormalizer()
encoder_normalizer = MinMaxFeatureNormalizer()

In [4]:
epochs = 1000
representiation_dim = 64
EXPERIMENT_NAME='100 world 1000 path overfit offset-encoding end2end encoder trianing'

In [5]:

CONFIG = {
    # Model configuration
    'representation_dim': representiation_dim, # Dimension of the representation
    'dim_mults':(1, 2, 4, 8), # Dimension multipliers for hidden layers of the model # (1, 4, 8)
    'attention':False, 
    
    
    
    # Encoder configuration
    'embeder_num_of_hidden_layers' : 1,
    'encoder_conditional_dim': representiation_dim//2,
    
    # Hyperparameters for training
    'batch_size': 8,
    'num_epochs': epochs,
    'ema': EMA(beta=0.99), # Exponential moving average for the model weights
        ## Optimizer configuration
        'optimizer': Prodigy,
        'optimizer_kwargs': {
            'lr': 1., # ! ONLY FOR PRODIGY OPTIMIZER
            'weight_decay': 0.01, 
            'safeguard_warmup':True,
            'use_bias_correction':True,
            'betas': (0.9, 0.99),
            },
        # Scheduler configuration
        'scheduler': torch.optim.lr_scheduler.CosineAnnealingLR,
        'scheduler_kwargs': {
            # 'gamma': 0.999,
            'T_max': 10, # Total number of iterations
        },
    
    'loss_weights':{
        'MSE': 0.3,
        'ObstacleFreePathLoss': 0.5,
        'GraphBasedConsistencyLoss':0.2,
    },
    # Hyperparameters for diffusion process
    'noise_steps': 256,
    'normalize': True,
    'normalizer':normalizer,
    'encoder_normalizer':encoder_normalizer,
    'cfg_scale': 3,


    # Dataset specific configuration
    'n_paths_per_world': 1000,
    'n_worlds': 100,
    'n_waypoints': 64, # due to the archtechture has to be a number that is a power of 2 
    'single_world_dataset':False,
}

# Dataloader

In [6]:
from dataset import RobotPathDataset

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
device

cuda:0


device(type='cuda', index=0)

### Normalizer

In [8]:
file = '/home/karim.samir.lotfy/tum-adlr-ss24-09/data/RobotPathData/SingleSphere02_all.db'
# file = '/home/karim.samir.lotfy/tum-adlr-ss24-09/data/SingleSphere02_one-world.db'
dataset = RobotPathDataset(file, n_paths_per_world=CONFIG['n_paths_per_world'], n_worlds=CONFIG['n_worlds'],  n_waypoints=CONFIG['n_waypoints'],   normalizer=CONFIG['normalizer'], single_world_dataset=CONFIG['single_world_dataset'])
print(f' sample shape {dataset[0]["path"].shape} with {len(dataset)} samples')
indx_sample = 5
sample = dataset[indx_sample]

world_indx: 36, world_distance_field_images: torch.Size([64, 64])
world_indx: 37, world_distance_field_images: torch.Size([64, 64])
world_indx: 65, world_distance_field_images: torch.Size([64, 64])
world_indx: 66, world_distance_field_images: torch.Size([64, 64])
world_indx: 109, world_distance_field_images: torch.Size([64, 64])
world_indx: 110, world_distance_field_images: torch.Size([64, 64])
world_indx: 124, world_distance_field_images: torch.Size([64, 64])
world_indx: 255, world_distance_field_images: torch.Size([64, 64])
world_indx: 266, world_distance_field_images: torch.Size([64, 64])
world_indx: 267, world_distance_field_images: torch.Size([64, 64])
world_indx: 361, world_distance_field_images: torch.Size([64, 64])
world_indx: 373, world_distance_field_images: torch.Size([64, 64])
world_indx: 422, world_distance_field_images: torch.Size([64, 64])
world_indx: 423, world_distance_field_images: torch.Size([64, 64])
world_indx: 447, world_distance_field_images: torch.Size([64, 64])

data shape: torch.Size([22000, 64, 2]), min_values: tensor([0, 0], device='cuda:0'), max_values: tensor([10, 10], device='cuda:0'), n_worlds:36, samples: 22000
 sample shape torch.Size([64, 2]) with 22000 samples


In [9]:
np.unique(dataset.worlds_indx)

array([ 36,  37,  65,  66, 109, 110, 124, 255, 266, 267, 361, 373, 422,
       423, 447, 450, 468, 469, 478, 479, 480, 572, 573, 620, 621, 622,
       623, 691, 692, 697, 872, 873, 916, 917, 929, 930])

In [10]:
denormalized_sample = normalizer.denormalize(sample['path'])
denormalized_sample

tensor([[7.0881, 5.2770],
        [6.9957, 5.3076],
        [6.9034, 5.3381],
        [6.8111, 5.3687],
        [6.7188, 5.3995],
        [6.6271, 5.4315],
        [6.5354, 5.4636],
        [6.4437, 5.4957],
        [6.3518, 5.5271],
        [6.2590, 5.5560],
        [6.1663, 5.5849],
        [6.0736, 5.6139],
        [5.9814, 5.6447],
        [5.8901, 5.6783],
        [5.7987, 5.7120],
        [5.7074, 5.7456],
        [5.6202, 5.7873],
        [5.5386, 5.8401],
        [5.4570, 5.8928],
        [5.3754, 5.9456],
        [5.3183, 6.0221],
        [5.2682, 6.1054],
        [5.2213, 6.1895],
        [5.2243, 6.2865],
        [5.2273, 6.3836],
        [5.2304, 6.4806],
        [5.2524, 6.5748],
        [5.2784, 6.6684],
        [5.3043, 6.7620],
        [5.3412, 6.8517],
        [5.3814, 6.9401],
        [5.4216, 7.0285],
        [5.4541, 7.1200],
        [5.4843, 7.2124],
        [5.5145, 7.3049],
        [5.5537, 7.3936],
        [5.5995, 7.4795],
        [5.6454, 7.5653],
        [5.6

In [11]:
WORLD_IMG = sample['world_img'] 

In [12]:
from torch.utils.data import DataLoader
batch_size = CONFIG['batch_size']
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)
# iterate over the dataset
# get the first batch
sample_batch = next(iter(dataloader))


In [13]:
sample_batch['world_indx'].shape

torch.Size([8])

## Loss Functions

### MSE

### ObstacleFreeLoss

In [14]:
from model import ObstacleFreePathLoss, GraphBasedConsistencyLoss
obstacle_avoidance_loss_fn = ObstacleFreePathLoss(normalizer=CONFIG['normalizer'])
graph_based_consistency_loss_fn = GraphBasedConsistencyLoss()

In [15]:
NOISE_TO_ADD_ON_PATH = 0.1
# This loss punishes the model for generating paths that collide with obstacles. 
for i in range(3):
    # create a subpolt for 2 graphs
    fig, axs = plt.subplots(1, 2, figsize=(16, 8))
    # the 1st will be the path and its distance field + loss value 
    indx = np.random.randint(len(dataset))
    sample = dataset[indx]
    dimg = sample['world_distance_field_img'].to('cuda')
    path = sample['path']
    path = path.unsqueeze(0)
    dimg = dimg.unsqueeze(0)
    l = obstacle_avoidance_loss_fn(path, dimg)
    dimg = sample['world_distance_field_img']
    path = sample['og_path'].cpu().detach().numpy()
    axs[0].imshow(dimg.T, origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
    axs[0].plot(path[:, 0], path[:, 1], '-o', color='red')
    axs[0].set_title(f'loss {l}')
    # the 2nd will be the path + some noise and its distance field + loss value
    sample = dataset[indx]
    dimg = sample['world_distance_field_img'].to('cuda')
    path = sample['path']
    path = path.unsqueeze(0)
    dimg = dimg.unsqueeze(0)
    noise = torch.randn_like(path) * NOISE_TO_ADD_ON_PATH
    path = path + noise
    l = obstacle_avoidance_loss_fn(path, dimg)
    dimg = sample['world_distance_field_img']
    path_vis = obstacle_avoidance_loss_fn.normalizer.denormalize(path).cpu().detach().numpy()
    axs[1].imshow(dimg.T, origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
    axs[1].plot(path_vis[0, :, 0], path_vis[0, :, 1], '-o', color='red')
    axs[1].set_title(f'loss {l}')


In [16]:
NOISE_TO_ADD_ON_PATH = 0.1
# This loss punishes the model for generating paths that collide with obstacles. 
for i in range(3):
    # create a subpolt for 2 graphs
    fig, axs = plt.subplots(1, 2, figsize=(16, 8))
    # the 1st will be the path and its distance field + loss value 
    indx = np.random.randint(len(dataset))
    sample = dataset[indx]
    dimg = sample['world_distance_field_img'].to('cuda')
    path = sample['path']
    path = path.unsqueeze(0)
    dimg = dimg.unsqueeze(0)
    l = graph_based_consistency_loss_fn(path)
    dimg = sample['world_distance_field_img']
    path = sample['og_path'].cpu().detach().numpy()
    axs[0].imshow(dimg.T, origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
    axs[0].plot(path[:, 0], path[:, 1], '-o', color='red')
    axs[0].set_title(f'loss {l}')
    # the 2nd will be the path + some noise and its distance field + loss value
    sample = dataset[indx]
    dimg = sample['world_distance_field_img'].to('cuda')
    path = sample['path']
    path = path.unsqueeze(0)
    dimg = dimg.unsqueeze(0)
    noise = torch.randn_like(path) * NOISE_TO_ADD_ON_PATH
    path = path + noise
    l = graph_based_consistency_loss_fn(path)
    dimg = sample['world_distance_field_img']
    path_vis = obstacle_avoidance_loss_fn.normalizer.denormalize(path).cpu().detach().numpy()
    axs[1].imshow(dimg.T, origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
    axs[1].plot(path_vis[0, :, 0], path_vis[0, :, 1], '-o', color='red')
    axs[1].set_title(f'loss {l}')

# Archtechture

### encoder

In [17]:
from model import WorldIndexEmbeder, FlattenEmbeder, VAEXperiment

In [18]:
# Import pretrained vae model
model_experiment = VAEXperiment.load_from_checkpoint(r'/home/karim.samir.lotfy/tum-adlr-ss24-09/lightning_logs/version_10/checkpoints/epoch=227-step=256500.ckpt')
model_experiment.freeze()
vae_model = model_experiment.model
vae_model.to(device)
vae_model.encoder.to(device)

Sequential(
  (0): Sequential(
    (0): Conv2d(1, 4, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
  )
  (1): Sequential(
    (0): Conv2d(4, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
  )
  (2): Sequential(
    (0): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
  )
  (3): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
  )
  (4): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(

In [19]:
from model import Encoder
class wrapper_encoder(Encoder):
    def __init__(self, vae_model):
        super(wrapper_encoder, self).__init__()
        self.vae_model = vae_model
    def forward(self, x):
        results = self.vae_model.encode(x)
        mu, log_var = results
        z = self.vae_model.reparameterize(mu, log_var)
        return z

    def decode(self, z):
        return self.vae_model.decode(z)

In [20]:
embedding_dim = CONFIG['encoder_conditional_dim']
# Define the model
sample_input = sample['world_distance_field_img']
encoder = wrapper_encoder(vae_model)

In [21]:
sample_batch = sample_batch
dimgs = sample_batch['world_distance_field_img'].to(device)
encoding = encoder(dimgs)

In [22]:

# assert encoding.shape[1] == CONFIG['encoder_conditional_dim']
encoding.shape

torch.Size([8, 64])

### Model and diffusion

In [23]:
from model import Diffusion, TemporalUnet

In [24]:
# Create the UNet
transition_dim = sample['path'].shape[-1] # transition_dim is the dimension the model should predict. not the timesteps
print(transition_dim)
model = TemporalUnet(horizon=CONFIG['noise_steps'], transition_dim=transition_dim, dim=CONFIG['representation_dim'], dim_mults=CONFIG['dim_mults'],encoder=encoder, attention=CONFIG['attention'],device=device)

2
[ models/temporal ] Channel dimensions: [(2, 64), (64, 128), (128, 256), (256, 512)]
[ models/temporal ] Embedding dimensions: env encoding 32, start_pos encoding 8, end_pos encoding 8, time encoding 16
[(2, 64), (64, 128), (128, 256), (256, 512)]


In [25]:
model.to(device);
model.device

device(type='cuda', index=0)

In [26]:
# create the diffusion model
diffusion = Diffusion(input_shape=sample['relative_path'].shape, noise_steps=CONFIG['noise_steps'], device=device)

In [27]:
# test the model
x = sample_batch
timesteps = diffusion.sample_timesteps(batch_size)
y = model(x['relative_path'], timesteps)
assert y.shape == (batch_size, CONFIG['n_waypoints'], 2)
y.shape

torch.Size([8, 64, 2])

In [28]:
# noise input 
timesteps = diffusion.sample_timesteps(batch_size)
# Create timesteps that are liner
timesteps = torch.linspace(1, CONFIG['noise_steps']-1, steps=batch_size, dtype=torch.int).to(device)
x_t, noise = diffusion.noise_input(sample_batch['relative_path'],timesteps)
x_t.shape, noise.shape

(torch.Size([8, 64, 2]), torch.Size([8, 64, 2]))

In [29]:
sample_batch.keys()

dict_keys(['path', 'og_path', 'world_indx', 'world_img', 'world_distance_field_img', 'relative_path', 'straight_line_path'])

In [30]:
# PLOT INPUT VS NOISED INPUT
# create a subpolt for 2 graphs
fig, axs = plt.subplots(batch_size, 3, figsize=(16, 16))
for i in range(batch_size):
    # the 1st will be the path and its distance field + loss value 
    path = sample_batch['path'][i].cpu().detach().numpy()
    axs[i, 0].plot(path[:, 0], path[:, 1], '-o', color='red')
    # the 2nd will be the path + some noise and its distance field + loss value
    path_noised = sample_batch['relative_path'][i].cpu().detach().numpy()
    axs[i, 1].plot(path_noised[:, 0], path_noised[:, 1], '-o', color='red')
    # the 3rd will be the path + some noise and its distance field + loss value
    path_noised = x_t[i].cpu().detach().numpy()
    axs[i, 2].plot(path_noised[:, 0], path_noised[:, 1], '-o', color='red')

In [31]:
import matplotlib.cm as cm

number_of_samples_to_show = 5
# test the model
x = sample_batch['path'][0:number_of_samples_to_show]
timesteps = diffusion.sample_timesteps(number_of_samples_to_show)
y = model(x, timesteps)
# noise input 
timesteps = diffusion.sample_timesteps(number_of_samples_to_show)
x_t, noise = diffusion.noise_input(x,timesteps)
x_t.shape, noise.shape
# plot sample entire batch
colors = cm.rainbow(np.linspace(0, 1, len(x)))
def visualize_diffusion_process(x, x_t, normalizer=None):
    if normalizer:
        x = normalizer.denormalize(x)
        x_t = normalizer.denormalize(x_t)
    x = x.cpu().detach().numpy()
    x_t = x_t.cpu().detach().numpy()
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    for i in range(number_of_samples_to_show):
        ax.plot(x[i, :, 0], x[i, :, 1], 'o-', label='input', color=colors[i])
        ax.plot(x_t[i, :, 0], x_t[i, :, 1], 'o-', label=f'noise t={timesteps[i]}', color=colors[i], alpha=0.5)
    plt.legend()
    plt.show()

print(timesteps)
# visualize_diffusion_process(x, x_t, normalizer=normalizer)
# visualize_diffusion_process(x, x_t, normalizer=None)


tensor([231,  56, 180, 211, 154], device='cuda:0')


# Training

### Set up validation and plotting function

In [32]:
# write training loop 
import torch.optim as optim
import torch.nn as nn
from utils.visualize.vae import plot_vae_reconstruciton
from utils.visualize.diffusion import plot_diffusions, plot_noise_and_predicted_noise

### Set up LoggingManager

In [33]:
from LoggingManager import LoggingManager
logging_manager = LoggingManager(experiement_name=EXPERIMENT_NAME)
logging_manager.log_config(CONFIG);

Logging to logs/100 world 1000 path overfit offset-encoding end2end encoder trianing/19h-09m-01-06-2024/logs.log


In [34]:
# log the model and input sample 
input_tensor = sample_batch['path']
cond = sample_batch['world_distance_field_img']
# logging_manager.tb_log_graph(model)

# Train 

In [35]:
START_FROM_CHECKPOINT = False
Checkpoint_file = r"logs/big-model+new obstacle loss/20h-39m-21-05-2024/model-epoch_4-loss_0.100-time_21h-22m-21-05-2024"

In [36]:
# train the model

import random


optimizer = CONFIG['optimizer'](model.parameters(), **CONFIG['optimizer_kwargs'])
scheduler = CONFIG['scheduler'](optimizer, **CONFIG['scheduler_kwargs'])
ema = CONFIG['ema']
ema_model = copy.deepcopy(model).eval().requires_grad_(False)
epoch = 0
# Restart from checkpoint 
if START_FROM_CHECKPOINT:
    model, optimizer, scheduler, epoch, loss = logging_manager.load_checkpoint(model, optimizer, scheduler, filename=Checkpoint_file)
    ema.step_ema(ema_model,model)
    logging_manager.log_message(f'Restarting from epoch {epoch} with loss {loss}')
    print(f'Restarting from epoch {epoch} with loss {loss}')


Using decoupled weight decay


In [37]:

for epoch in range(epoch, CONFIG['num_epochs']):
    pbar = tqdm.tqdm(dataloader, total=len(dataloader), leave=True)
    running_loss = 0.0
    # iterate over the dataset
    for i, batch_and_info in enumerate(pbar):
        batch, batch_world_indx, batch_world_img, batch_world_distance_field_img, batch_relative, batch_straight_line = batch_and_info['path'], batch_and_info['world_indx'], batch_and_info['world_img'], batch_and_info['world_distance_field_img'], batch_and_info['relative_path'], batch_and_info['straight_line_path']
        
        
        # Training step
        model.train()
        optimizer.zero_grad()
        x = batch_relative.to(device)
        cond = batch_world_distance_field_img.to(device)
        start_pos = x[:, 0, :]
        end_pos = x[:, -1, :]
        # if random.random() < 0.1: # in 10% of the time. sample unconditional. This is to use CFG
        #     cond = None
        # preturb input. add noise to it
        timesteps_choosen = diffusion.sample_timesteps(x.shape[0])
        x_t, noise = diffusion.noise_input(x,timesteps_choosen)
        # get model to predict the noise
        pred_noise = model(x_t, timesteps_choosen, cond, start_pos, end_pos)
        # pred_relative_path = model(x_t, timesteps_choosen, cond, start_pos, end_pos)
        pred_relative_path = x + pred_noise
        pred_path = batch_straight_line + pred_relative_path
            
        # Loss 
        loss_noise_prediction =nn.SmoothL1Loss()(noise, pred_noise)
        # loss_noise_prediction = nn.SmoothL1Loss()(pred_relative_path, x)
        loss_obstacle_avoidance =obstacle_avoidance_loss_fn(pred_path, cond)
        loss_graph_based_consistency =graph_based_consistency_loss_fn(pred_path)
        loss = CONFIG['loss_weights']['MSE']*loss_noise_prediction + CONFIG['loss_weights']['ObstacleFreePathLoss']*loss_obstacle_avoidance +  CONFIG['loss_weights']['GraphBasedConsistencyLoss'] * loss_graph_based_consistency
        loss.backward()
        optimizer.step()
        # update the EMA model
        ema.step_ema(ema_model,model)
        running_loss += loss.item()
        logging_manager.plot_losses(loss.item())
        logging_manager.tb_scalar('Loss/MSE', loss_noise_prediction.item())
        logging_manager.tb_scalar('Loss/ObstacleAvoidance', loss_obstacle_avoidance.item())
        logging_manager.tb_scalar('Loss/GraphBasedConsistency', loss_graph_based_consistency.item())
        pbar.set_postfix({'loss': loss.item()})
    
        if i % 100 == 0:

            # plot noise and predicted noise
            plot = plot_noise_and_predicted_noise(x, noise, pred_noise, timesteps_choosen, normalizer=CONFIG['normalizer'])
            logging_manager.tb_log_figure(plot, 'noise_and_predicted_noise', step= i + epoch * len(dataloader))

            # plot noise and predicted noise ema
            ema_model.eval()
            pred_noise_ema = ema_model(x_t, timesteps_choosen)
            plot = plot_noise_and_predicted_noise(x, noise, pred_noise_ema, timesteps_choosen, normalizer=CONFIG['normalizer'])
            logging_manager.tb_log_figure(plot, 'ema_noise_and_predicted_noise')
            
            # Log optimizer parameters and scheduler parameters
            logging_manager.tb_scalar('optimizer/learning_rate', optimizer.param_groups[0]['lr'])
            logging_manager.tb_scalar('optimizer/d', optimizer.param_groups[0]['d'])
            logging_manager.tb_scalar('optimizer/scheduler-lr', scheduler.get_lr()[0])
            
            # plot histogram of pred noise and noise 
            logging_manager.tb_log_histogram(noise, 'dist/noise')    
            logging_manager.tb_log_histogram(pred_noise, 'dist/pred_noise')
        
            
            # plot reconstruction of encoder 
            sample = cond
            # sample = sample.to(autoencoder.curr_device)
            recon,input_img, mu, var = model.encoder.vae_model.forward(sample)
            plot = plot_vae_reconstruciton(sample, recon)
            logging_manager.tb_log_figure(plot, 'vae_reconstruction')

            

            
    
    logging_manager.log_message(f'Epoch {epoch} Epoch loss: {running_loss/len(dataloader)}')
    print(f'Epoch {epoch} Epoch loss: {running_loss/len(dataloader)}')
    # scheduler
    scheduler.step()
    running_loss = 0.0
    # Validation 
    model.eval()




    if epoch% 2 == 0:
        logging_manager.save_checkpoint(model, optimizer, scheduler,epoch, loss)

        if epoch > 1:
            # unfreeze the encoder
            model.encoder.requires_grad_(True)
        # # sample model
        # sample from paths 
        n = min(8, len(batch))
        paths = batch_relative[:n]
        cond = batch_world_distance_field_img[:n]
        true_paths = paths + batch_straight_line[:n]
        samples_offset , intermediates_offset = diffusion.sample_from_paths(model, n=n, paths=paths, cond=cond, normalizer=CONFIG['normalizer'])
        # intermediates.shape = (diffusion_steps_saved, batch, n_waypoints, 2)
        # make offset paths into abosolute paths in intermediates, by adding straightline now the offset is the path
        intermediates = batch_straight_line[:n].detach().cpu().numpy() + intermediates_offset
        samples = batch_straight_line[:n].detach().cpu().numpy() + samples_offset
        plot = plot_diffusions(intermediates, world_imgs=batch_world_img, normalizer=CONFIG['normalizer'], true_paths=true_paths);
        logging_manager.tb_log_figure(plot, 'diffusion_samples')
        # samples, intermediates = diffusion.sample(ema_model,5, cfg_scale=CONFIG['cfg_scale'])
        # plot = plot_diffusions(intermediates, normalizer=CONFIG['normalizer'])
        # logging_manager.tb_log_figure(plot, 'diffusion_samples_ema_cfg', step=epoch)
        for k in range(1):
            
            last_index_for_waypoints = CONFIG['n_waypoints'] - 1
            constraints = {
                0: torch.tensor([0, 0], device=device),
                last_index_for_waypoints: torch.tensor([9, 9],device=device),
            }
            n=CONFIG['batch_size']
            cond = batch_world_distance_field_img.to(device)
            world_imgs = batch_world_img
            # logging_manager.log_message(f'Generating samples from edge to edge with worlds {batch_world_indx}')
            # samples, intermediates = diffusion.sample_with_constraints(model, n=n, constraints=constraints, cond=cond,normalizer=CONFIG['normalizer'])
            # plot = plot_diffusions(intermediates, world_imgs=world_imgs, normalizer=CONFIG['normalizer']);
            # logging_manager.tb_log_figure(plot, f'samples edge to edge World {k}', step=epoch);
            # sample ema model with cfg
            # samples, intermediates = diffusion.sample_with_constraints(ema_model, n=n, constraints=constraints, cond=cond, normalizer=CONFIG['normalizer'], cfg_scale=CONFIG['cfg_scale'])
            # plot = plot_diffusions(intermediates, world_imgs=world_imgs, normalizer=CONFIG['normalizer']);
            # logging_manager.tb_log_figure(plot, f'samples edge to edge World {k} ema+cfg', step=epoch);
            
        # log model embedding space
        # features = model.get_last_embedding()
        # metadata = batch_world_indx
        # metadata should be a list of strings
        # metadata = [f'world {str(i)}' for i in metadata]
        # images = batch_world_img # batch * 64 * 64
        # images = images.unsqueeze(1) # turn to batch * C * H * W
        # logging_manager.tb_log_embedding_space(features, metadata, images)
        
        # # log model weights
        # last_layer_weights = model.final_conv[-1].weight
        # logging_manager.tb_log_histogram(last_layer_weights, 'final layer wieghts')
        # # log model embedding space weights
        # embedding_weights = model.encoder.embedding[-2].weight
        # logging_manager.tb_log_histogram(embedding_weights, 'embedding layer wieghts')
 
        
        
print('Finished Training')    

  0%|          | 0/2750 [00:00<?, ?it/s]

100%|██████████| 2750/2750 [06:06<00:00,  7.51it/s, loss=0.101]   


Epoch 0 Epoch loss: 0.09921330880991776


255it [00:03, 66.51it/s]
100%|██████████| 2750/2750 [06:15<00:00,  7.32it/s, loss=0.112]   
INFO:LoggingManager:Epoch 1 Epoch loss: 0.06211689284884117


Epoch 1 Epoch loss: 0.06211689284884117


...

Epoch 16 Epoch loss: 0.041424338369714944


255it [00:03, 67.98it/s]
100%|██████████| 2750/2750 [06:28<00:00,  7.07it/s, loss=0.0246]   
INFO:LoggingManager:Epoch 17 Epoch loss: 0.042609245207499376


Epoch 17 Epoch loss: 0.042609245207499376


 77%|███████▋  | 2118/2750 [05:00<01:29,  7.06it/s, loss=0.092]    


KeyboardInterrupt: 

# Sample Model

In [38]:
logging_manager.save_checkpoint(model, optimizer, scheduler,epoch, loss)

INFO:LoggingManager:Checkpoint saved to logs/100 world 1000 path overfit offset-encoding end2end encoder trianing/19h-09m-01-06-2024/model-epoch_18-loss_0.092-time_21h-09m-01-06-2024
